In [1]:
device = "0"
model_name = "meta-llama/Llama-3.2-3B"
dataset_size = 200
evaluate_scores = True

In [2]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('../src')

In [3]:
import torch
from utils import load_model_and_tokenizer, load_c4

experiment = "varying_length"

device = torch.device(f"cuda:{device}" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model, tokenizer = load_model_and_tokenizer(model_name, device)

data = load_c4(tokenizer, dataset_size)
data[0]

/home/ubuntu/anaconda3/envs/wepa/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda:0


`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 2/2 [00:02<00:00,  1.16s/it]


Model: meta-llama/Llama-3.2-3B
Loading dataset from cache...


{'input_ids': tensor([[128000,    791,  59796,   6406,   8925,   6787,    311,   7055,   7884,
          13658,    389,   7231,   8339,    400,   1419,   3610,    323,   1023,
          36580,    311,   1977,    264,  26097,  15679,    304,  29016,   4409,
             11,    719,   1193,   1306,  11011,  12483,    315,  18671,  13286,
          11062,    323,  28424,  49262,    369,    477,   2403,    279,   2447,
            627,    791,   4330,  44650,   4580]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1]])}

In [4]:
from utils import (
    load_default_wepa_watermarker,
    load_default_exp_watermarker,
    load_default_kgw_watermarker,
    load_default_unbiased_watermarker,
    get_wat_name,
    run_experiment,
)

wats = [
    load_default_wepa_watermarker(
        model.config.vocab_size,
        device,
        degree=1,
    ),
    load_default_wepa_watermarker(
        model.config.vocab_size,
        device,
        degree=2,
    ),
    load_default_exp_watermarker(
        model.config.vocab_size,
        device,
    ),
    load_default_kgw_watermarker(
        model.config.vocab_size,
        device,
    ),
    load_default_unbiased_watermarker(
        model.config.vocab_size,
        device,
    ),
]

max_lengths1 = [2, 4, 6, 8, 10, 12, 14, 16, 18, 20]
max_lengths2 = [5, 10, 15, 20, 25, 30, 35, 40, 45, 50]
max_lengths = max_lengths1 + max_lengths2

for wat in wats:
    for max_length in max_lengths:
        wat_name = get_wat_name(wat)

        run_experiment(
            experiment=experiment,
            run_name=f"{model.name_or_path}_{wat_name}",
            wat=wat,
            model=model,
            tokenizer=tokenizer,
            data=data,
            max_length=max_length,
            device=device,
            evaluate_scores=evaluate_scores,
        )

Running experiment: <watermarker.wepa.WepaWatermarker object at 0x7f9e147641d0>


  6%|▌         | 12/200 [00:09<02:25,  1.29it/s]


KeyboardInterrupt: 